# 🏁 Formula 1 Workshop: Data Setup

*This notebook downloads Formula 1 datasets and loads them into a Databricks volume.*

---

## Step 1: Connect to Compute
Connect to serverless compute in the compute drop-down at your top right.

In [0]:
# Download Formula 1 CSV Files directly to Volume
import urllib.request
import os

# Create the main catalog if it doesn't exist
spark.sql("CREATE CATALOG IF NOT EXISTS main")

# Create the volume if it doesn't exist
spark.sql("CREATE VOLUME IF NOT EXISTS main.default.formula1")
print("✓ Volume created/verified")

# Create the separate volume for streaming
spark.sql("CREATE VOLUME IF NOT EXISTS main.default.streaming_formula1")
print("✓ Streaming volume created/verified")

# Define the files to download
files_to_download = {
    'Formula1_2026Season_QualifyingResults.csv': 'https://raw.githubusercontent.com/toUpperCase78/formula1-datasets/master/Formula1_2026Season_QualifyingResults.csv',
    'Formula1_2026Season_RaceResults.csv': 'https://raw.githubusercontent.com/toUpperCase78/formula1-datasets/master/Formula1_2026Season_RaceResults.csv',
    'Formula1_2026Season_SprintQualifyingResults.csv': 'https://raw.githubusercontent.com/toUpperCase78/formula1-datasets/master/Formula1_2026Season_SprintQualifyingResults.csv',
    'Formula1_2026Season_SprintResults.csv': 'https://raw.githubusercontent.com/toUpperCase78/formula1-datasets/master/Formula1_2026Season_SprintResults.csv'
}

# Download each file to the volume
volume_path = '/Volumes/main/default/formula1/'

for filename, url in files_to_download.items():
    try:
        print(f"Downloading {filename}...")
        file_path = volume_path + filename
        urllib.request.urlretrieve(url, file_path)
        print(f"✓ Successfully downloaded {filename}")
    except Exception as e:
        print(f"✗ Error downloading {filename}: {str(e)}")

# Download SprintQualifyingResults.csv to streaming_formula1 volume
streaming_volume_path = '/Volumes/main/default/streaming_formula1/'
streaming_filename = 'Formula1_2026Season_SprintQualifyingResults.csv'
streaming_url = 'https://raw.githubusercontent.com/toUpperCase78/formula1-datasets/master/Formula1_2026Season_SprintQualifyingResults.csv'
try:
    print(f"Downloading {streaming_filename} to streaming volume...")
    streaming_file_path = streaming_volume_path + streaming_filename
    urllib.request.urlretrieve(streaming_url, streaming_file_path)
    print(f"✓ Successfully downloaded {streaming_filename} to streaming volume")
except Exception as e:
    print(f"✗ Error downloading {streaming_filename} to streaming volume: {str(e)}")

print("\n" + "="*50)
print("Download complete! Listing files in volume:")
print("="*50)

# List files in the volume
try:
    files = dbutils.fs.ls(volume_path)
    for file in files:
        if file.name.endswith('.csv'):
            print(f"✓ {file.name} ({file.size} bytes)")
except Exception as e:
    print(f"Error listing files: {e}")

print("\nFormula 1 data files are now ready in the volume!")

## ⚠️ Troubleshooting: Can't use the `main` catalog?

If `CREATE CATALOG IF NOT EXISTS main` fails because you don't have permission to create or use the `main` catalog, you have two options:

1. **Ask your admin** for a catalog you can create schemas/tables in (e.g. `bobby_dev`), OR
2. **Create your own catalog** if you have permission: `CREATE CATALOG IF NOT EXISTS your_catalog_name`.

Then **every other notebook in this workshop** needs to point at that catalog instead of `main`. Doing this by hand is tedious — let Genie Code rewrite the notebooks for you.

### 🧞 Use Genie Code to swap the catalog everywhere

1. Open the **Genie Code agent** from the notebook side panel (the ✨ icon).
2. Paste the prompt below, **replacing `YOUR_CATALOG_NAME`** with the catalog you want to use.
3. Let it run — it will rewrite every notebook in this folder so all `main.default.*` references use your catalog instead.

```
Replace every reference to the `main` catalog in every notebook in this workshop folder with `YOUR_CATALOG_NAME`.

Specifically:
- In SQL: `main.default.foo` → `YOUR_CATALOG_NAME.default.foo`
- In `read_files('/Volumes/main/default/...')`: swap the catalog segment to `YOUR_CATALOG_NAME`
- In `spark.sql("CREATE VOLUME IF NOT EXISTS main.default.formula1")` and similar: swap `main` to `YOUR_CATALOG_NAME`
- In `dbutils.fs.ls(...)` Volume paths: swap the catalog segment

Do NOT change the schema (`default`) or table/volume names — only the catalog.
Apply this across all `.ipynb`, `.sql`, and `.py` files in the current folder.
After the changes, list each file you edited.
```

> 💡 Once you've used Genie Code to update the notebooks, re-run this `00_Setup` notebook to create the volumes and download the data into your catalog.

## What's Next?

Now that you've set up the Formula 1 data in your Databricks workspace:

1. The data is available in the volume at `/Volumes/main/default/formula1/`
2. You can use these files in the workshop notebooks
3. Continue to the next notebook to start exploring the Databricks platform